# massive in silico screening with Prophet

This notebook demonstrates how to make predictions with Prophet with any of the checkpoints we have made available.

In [1]:
import pandas as pd
import yaml
from prophet import Prophet, set_config

/ictstr01/groups/ml01/workspace/alejandro.tejada/micromamba/envs/prophet/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load in the same config file that was used for finetuning, for the embedding files.

In [2]:
with open("config_file_finetuning.yaml", "r") as f:
    config = set_config(yaml.safe_load(f))

In [3]:
pretrained_checkpoint_path = "./ckpts/epoch=1-step=248.ckpt"
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=None,
    model_pth=pretrained_checkpoint_path,
)

returning trained model!
Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=512, out_features=1, bias=True)
)


Suppose we have some small molecules, some cell lines we would like to test them in, and we're interested in measuring their relative IC50. We can pass in lists of these inputs, and Prophet will return predictions for all combinations:

### Making predictions by passing all treatments, cell lines, and phenotypes you want to run

This format can be useful when running large combinatorial screens in silico, as it splits the experiments up into batches to help prevent memory errors.

In [4]:
iv_list = [
    "oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc4=c(c=c(c=c4)i)f)=o",
    "cc(nc1=cc=cc(n(c2=o)c(c(c(n2c3cc3)=o)=c(n4c)nc5=cc=c(c=c5f)i)=c(c4=o)c)=c1)=o",
    "fc1=cc=c(c(f)=c1c(c2=cnc3=nc=c(c=c32)c4=cc=c(c=c4)cl)=o)ns(ccc)(=o)=o",
    "cs(=o)c",  # DMSO
]
cl_list = [
    "A375",
    "UACC62",
    "WM983B",
    "MALME3M",
    "A2058",
    "WM793",
    "HT144",
    "RPMI7951",
    "SKMEL2",
    "SKMEL1",
    "HMCB",
    "MDAMB435S",
    "WM1799",
    "LOXIMVI",
]
ph_list = ["GDSC"]

In [5]:
# predict with lists of treatments and cell lines
df = model.predict(
    target_ivs=iv_list,
    target_cls=cl_list,
    target_phs=ph_list,
    iv_col=["iv1", "iv2"],  # pass to turn on combinatorial predictions
    num_iterations=1,
    save=False,
)
df

There are 1 iterations



  0%|                                                                                                                                                           | 0/1 [00:00<?, ?it/s]

Removing 0 such as [] from ['iv1', 'iv2']. 140 rows remaining.
Removing 0 such as [] from ['cell_line']. 140 rows remaining.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB MIG 3g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-c51c82f2-7e04-56a8-8b74-4211c9821715]


Predicting DataLoader 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  0.94it/s]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:06<00:00,  6.88s/it]


,iv1,iv2,cell_line,phenotype,iv1+iv2,value,pred
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.480913
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.484669
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.491997
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.473723
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.489981
...,...,...,...,...,...,...,...
135,cs(=o)c,cs(=o)c,SKMEL1,GDSC,cs(=o)c+cs(=o)c,_,0.513998
136,cs(=o)c,cs(=o)c,HMCB,GDSC,cs(=o)c+cs(=o)c,_,0.540128
137,cs(=o)c,cs(=o)c,MDAMB435S,GDSC,cs(=o)c+cs(=o)c,_,0.538795
138,cs(=o)c,cs(=o)c,WM1799,GDSC,cs(=o)c+cs(=o)c,_,0.526665


### Making predictions for a specific set of treatments, cell lines, and phenotypes

If we're interested in only a subset of the experimental matrix, we can also pass in a custom dataframe. (This is the recommended usage, as users understand exactly the list being predicted.)

In [6]:
# Construct a dataframe containing the experiments we want to run. In practice, the user would load
# a premade dataframe here.
input_df = pd.MultiIndex.from_product(
    [
        iv_list,
        cl_list,
    ],
    names=["iv1", "cell_line"],
)
input_df = input_df.to_frame(index=False).reset_index(drop=True)
input_df["iv2"] = "cs(=o)c"  # DMSO
input_df["phenotype"] = "GDSC"
input_df

,iv1,cell_line,iv2,phenotype
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,cs(=o)c,GDSC
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,cs(=o)c,GDSC
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,cs(=o)c,GDSC
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,cs(=o)c,GDSC
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,cs(=o)c,GDSC
5,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM793,cs(=o)c,GDSC
6,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,HT144,cs(=o)c,GDSC
7,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,RPMI7951,cs(=o)c,GDSC
8,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,SKMEL2,cs(=o)c,GDSC
9,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,SKMEL1,cs(=o)c,GDSC


In [7]:
df = model.predict(input_df, num_iterations=1, save=False)
df

There are 1 iterations


  0%|                                                                                                                                                           | 0/1 [00:00<?, ?it/s]GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-c51c82f2-7e04-56a8-8b74-4211c9821715]


Predicting DataLoader 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 28.32it/s]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.28s/it]


,iv1,cell_line,iv2,phenotype,pred
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,cs(=o)c,GDSC,0.489043
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,cs(=o)c,GDSC,0.495842
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,cs(=o)c,GDSC,0.501370
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,cs(=o)c,GDSC,0.484853
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,cs(=o)c,GDSC,0.499624
5,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM793,cs(=o)c,GDSC,0.493993
6,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,HT144,cs(=o)c,GDSC,0.495368
7,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,RPMI7951,cs(=o)c,GDSC,0.479674
8,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,SKMEL2,cs(=o)c,GDSC,0.506403
9,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,SKMEL1,cs(=o)c,GDSC,0.471883
